# NumPy Mastery for AI

**Module 01: Python for AI**  
**Objective**: Master NumPy for efficient numerical computing in AI/ML

NumPy is the foundation of the Python scientific computing stack. Understanding NumPy deeply is crucial for:
- Implementing ML algorithms from scratch
- Understanding tensor operations in PyTorch/TensorFlow
- Writing efficient, vectorized code
- Debugging numerical issues

## What You'll Learn
1. Array creation and manipulation
2. Broadcasting mechanics
3. Vectorization techniques
4. Linear algebra operations
5. Performance optimization
6. Memory layout (C vs Fortran order)
7. Advanced indexing and slicing

## 1. Array Creation and Basics

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
from typing import Tuple

# Set random seed for reproducibility
np.random.seed(42)

print(f"NumPy version: {np.__version__}")

### 1.1 Different Ways to Create Arrays

In [ ]:
# From lists
a = np.array([1, 2, 3, 4, 5])
print(f"1D array: {a}")
print(f"Shape: {a.shape}, Dtype: {a.dtype}, Ndim: {a.ndim}")

# 2D arrays
b = np.array([[1, 2, 3], [4, 5, 6]])
print(f"\n2D array:\n{b}")
print(f"Shape: {b.shape}, Size: {b.size}, Dtype: {b.dtype}")

# Specifying dtype
c = np.array([1, 2, 3], dtype=np.float32)
print(f"\nFloat32 array: {c}, Dtype: {c.dtype}")

# Common array creation functions
zeros = np.zeros((3, 4))
ones = np.ones((2, 3))
full = np.full((2, 2), 7)
identity = np.eye(4)
arange = np.arange(0, 10, 2)  # start, stop, step
linspace = np.linspace(0, 1, 5)  # start, stop, num_points

print(f"\nZeros (3x4):\n{zeros}")
print(f"\nOnes (2x3):\n{ones}")
print(f"\nFull (2x2) with 7:\n{full}")
print(f"\nIdentity (4x4):\n{identity}")
print(f"\nArange [0, 10) step 2: {arange}")
print(f"\nLinspace [0, 1] 5 points: {linspace}")

### 1.2 Random Arrays (Critical for ML)

In [ ]:
# Random arrays with different distributions
uniform = np.random.rand(3, 3)  # Uniform [0, 1)
normal = np.random.randn(3, 3)  # Standard normal N(0, 1)
randint = np.random.randint(0, 10, size=(3, 3))  # Random integers

# Custom distributions
custom_normal = np.random.normal(loc=5.0, scale=2.0, size=(3, 3))
custom_uniform = np.random.uniform(low=-1, high=1, size=(3, 3))

print(f"Uniform [0, 1):\n{uniform}\n")
print(f"Standard Normal:\n{normal}\n")
print(f"Random Integers [0, 10):\n{randint}\n")
print(f"Custom Normal (μ=5, σ=2):\n{custom_normal}\n")
print(f"Custom Uniform [-1, 1):\n{custom_uniform}\n")

# Xavier/He initialization (common in deep learning)
def xavier_init(shape: Tuple[int, int]) -> np.ndarray:
    """Xavier initialization: W ~ N(0, sqrt(2/(n_in + n_out)))"""
    n_in, n_out = shape
    std = np.sqrt(2.0 / (n_in + n_out))
    return np.random.normal(0, std, size=shape)

def he_init(shape: Tuple[int, int]) -> np.ndarray:
    """He initialization: W ~ N(0, sqrt(2/n_in))"""
    n_in, n_out = shape
    std = np.sqrt(2.0 / n_in)
    return np.random.normal(0, std, size=shape)

xavier_weights = xavier_init((100, 50))
he_weights = he_init((100, 50))

print(f"Xavier init (100→50): mean={xavier_weights.mean():.6f}, std={xavier_weights.std():.6f}")
print(f"He init (100→50): mean={he_weights.mean():.6f}, std={he_weights.std():.6f}")

## 2. Broadcasting: The Secret to Vectorization

Broadcasting is NumPy's mechanism for performing operations on arrays of different shapes.

**Rules**:
1. If arrays have different ranks, prepend 1s to the shape of the lower-rank array
2. Arrays are compatible if dimensions are equal or one is 1
3. After broadcasting, each array behaves as if it had the larger shape

In [ ]:
# Example 1: Scalar broadcasting
a = np.array([1, 2, 3, 4])
b = 10
result = a + b  # b broadcasts to [10, 10, 10, 10]
print(f"[1,2,3,4] + 10 = {result}")

# Example 2: 1D + 1D broadcasting
a = np.array([1, 2, 3])
b = np.array([10, 20, 30])
result = a + b
print(f"\n[1,2,3] + [10,20,30] = {result}")

# Example 3: 2D + 1D broadcasting (column-wise)
a = np.array([[1, 2, 3],
              [4, 5, 6]])
b = np.array([10, 20, 30])
result = a + b  # b broadcasts to [[10,20,30], [10,20,30]]
print(f"\n2D + 1D (row broadcast):\n{result}")

# Example 4: 2D + 1D broadcasting (row-wise)
a = np.array([[1, 2, 3],
              [4, 5, 6]])
b = np.array([[10],
              [20]])  # Shape (2, 1)
result = a + b  # b broadcasts to [[10,10,10], [20,20,20]]
print(f"\n2D + 1D (column broadcast):\n{result}")

# Example 5: Outer product via broadcasting
a = np.array([1, 2, 3, 4])[:, np.newaxis]  # Shape (4, 1)
b = np.array([10, 20, 30])  # Shape (3,)
result = a * b  # Broadcasts to (4, 3)
print(f"\nOuter product:\n{result}")

# Example 6: Batch normalization (practical example)
def batch_norm_manual(X: np.ndarray) -> np.ndarray:
    """
    Normalize batch: X shape (batch_size, features)
    Normalize along batch dimension (axis=0)
    """
    mean = X.mean(axis=0, keepdims=True)  # Shape (1, features)
    std = X.std(axis=0, keepdims=True)    # Shape (1, features)
    X_norm = (X - mean) / (std + 1e-8)    # Broadcasting!
    return X_norm

X = np.random.randn(100, 5)  # 100 samples, 5 features
X_normalized = batch_norm_manual(X)
print(f"\nBatch normalization:")
print(f"Original - Mean: {X.mean(axis=0)}, Std: {X.std(axis=0)}")
print(f"Normalized - Mean: {X_normalized.mean(axis=0)}, Std: {X_normalized.std(axis=0)}")

## 3. Vectorization: Avoiding Loops

Vectorization is replacing explicit loops with array operations. It's 10-100x faster for large arrays.

In [ ]:
# Example 1: Element-wise operations
n = 1_000_000

# Slow: Python loop
a_list = list(range(n))
b_list = list(range(n))
start = time.time()
c_list = [a_list[i] + b_list[i] for i in range(n)]
time_loop = time.time() - start

# Fast: NumPy vectorization
a_np = np.arange(n)
b_np = np.arange(n)
start = time.time()
c_np = a_np + b_np
time_vectorized = time.time() - start

print(f"Loop time: {time_loop:.4f}s")
print(f"Vectorized time: {time_vectorized:.4f}s")
print(f"Speedup: {time_loop/time_vectorized:.1f}x")

# Example 2: Computing distances (common in ML)
def euclidean_distance_loop(X: np.ndarray, Y: np.ndarray) -> np.ndarray:
    """
    Compute pairwise Euclidean distances using loops.
    X: (n, d), Y: (m, d) -> distances: (n, m)
    """
    n, d = X.shape
    m, _ = Y.shape
    distances = np.zeros((n, m))
    for i in range(n):
        for j in range(m):
            distances[i, j] = np.sqrt(np.sum((X[i] - Y[j])**2))
    return distances

def euclidean_distance_vectorized(X: np.ndarray, Y: np.ndarray) -> np.ndarray:
    """
    Vectorized version using broadcasting.
    
    ||x - y||^2 = ||x||^2 + ||y||^2 - 2<x, y>
    """
    # X: (n, d), Y: (m, d)
    X_sq = np.sum(X**2, axis=1, keepdims=True)  # (n, 1)
    Y_sq = np.sum(Y**2, axis=1, keepdims=True)  # (m, 1)
    XY = X @ Y.T  # (n, m)
    
    distances_sq = X_sq + Y_sq.T - 2 * XY  # Broadcasting!
    distances_sq = np.maximum(distances_sq, 0)  # Numerical stability
    return np.sqrt(distances_sq)

# Test on small data
X = np.random.randn(100, 50)
Y = np.random.randn(80, 50)

start = time.time()
dist_loop = euclidean_distance_loop(X, Y)
time_loop = time.time() - start

start = time.time()
dist_vec = euclidean_distance_vectorized(X, Y)
time_vec = time.time() - start

print(f"\nDistance computation (100×80 points, 50 dims):")
print(f"Loop: {time_loop:.4f}s")
print(f"Vectorized: {time_vec:.4f}s")
print(f"Speedup: {time_loop/time_vec:.1f}x")
print(f"Results match: {np.allclose(dist_loop, dist_vec)}")

## 4. Advanced Indexing and Slicing

In [ ]:
# Basic slicing
a = np.arange(20).reshape(4, 5)
print(f"Original array:\n{a}\n")

# Slicing (returns views, not copies)
print(f"First two rows: \n{a[:2]}\n")
print(f"Every other row: \n{a[::2]}\n")
print(f"Second column: {a[:, 1]}\n")
print(f"Sub-array [1:3, 2:4]:\n{a[1:3, 2:4]}\n")

# Boolean indexing (returns copies)
mask = a > 10
print(f"Mask (>10):\n{mask}\n")
print(f"Elements > 10: {a[mask]}\n")

# Fancy indexing (integer arrays)
rows = np.array([0, 2, 3])
cols = np.array([1, 3, 4])
print(f"Fancy indexing a[[0,2,3], [1,3,4]]: {a[rows, cols]}\n")

# Combining techniques
print(f"Rows 0,2 all columns:\n{a[[0, 2], :]}\n")

# Assignment with indexing
b = a.copy()
b[b > 10] = 0  # Set all values > 10 to 0
print(f"After setting >10 to 0:\n{b}\n")

# np.where (ternary operator)
c = np.where(a > 10, 1, 0)  # If >10: 1, else: 0
print(f"Where >10: 1, else: 0:\n{c}")

## 5. Linear Algebra Operations

In [ ]:
# Matrix multiplication
A = np.random.randn(3, 4)
B = np.random.randn(4, 5)
C = A @ B  # or np.dot(A, B) or np.matmul(A, B)
print(f"Matrix multiplication (3x4) @ (4x5) = {C.shape}\n")

# Element-wise vs matrix multiplication
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
print(f"Element-wise multiply:\n{A * B}\n")
print(f"Matrix multiply:\n{A @ B}\n")

# Transpose
A = np.random.randn(3, 4)
print(f"A.shape: {A.shape}, A.T.shape: {A.T.shape}\n")

# Inverse (for square matrices)
A = np.random.randn(4, 4)
A_inv = np.linalg.inv(A)
print(f"A @ A_inv ≈ I:\n{np.round(A @ A_inv, 2)}\n")

# Solve linear system Ax = b
A = np.array([[3, 1], [1, 2]])
b = np.array([9, 8])
x = np.linalg.solve(A, b)
print(f"Solve Ax=b: x = {x}")
print(f"Verify Ax = {A @ x} (should equal b = {b})\n")

# Eigenvalues and eigenvectors
A = np.array([[4, -2], [1, 1]])
eigenvalues, eigenvectors = np.linalg.eig(A)
print(f"Eigenvalues: {eigenvalues}")
print(f"Eigenvectors:\n{eigenvectors}\n")

# SVD (Singular Value Decomposition)
A = np.random.randn(4, 6)
U, S, Vt = np.linalg.svd(A, full_matrices=False)
print(f"SVD: A = U @ diag(S) @ Vt")
print(f"U.shape: {U.shape}, S.shape: {S.shape}, Vt.shape: {Vt.shape}")
print(f"Reconstruction error: {np.linalg.norm(A - U @ np.diag(S) @ Vt):.2e}\n")

# Norms
v = np.array([3, 4])
print(f"L1 norm: {np.linalg.norm(v, ord=1)}")  # |3| + |4| = 7
print(f"L2 norm: {np.linalg.norm(v, ord=2)}")  # sqrt(9 + 16) = 5
print(f"L-inf norm: {np.linalg.norm(v, ord=np.inf)}")  # max(|3|, |4|) = 4

## 6. Reduction Operations

In [ ]:
# Create sample data
X = np.random.randn(5, 4)
print(f"Data (5 samples, 4 features):\n{X}\n")

# Along different axes
print(f"Sum all elements: {X.sum()}")
print(f"Sum along axis=0 (column sums): {X.sum(axis=0)}")
print(f"Sum along axis=1 (row sums): {X.sum(axis=1)}\n")

print(f"Mean all: {X.mean():.3f}")
print(f"Mean along axis=0: {X.mean(axis=0)}")
print(f"Mean along axis=1: {X.mean(axis=1)}\n")

print(f"Std all: {X.std():.3f}")
print(f"Std along axis=0: {X.std(axis=0)}")
print(f"Std along axis=1: {X.std(axis=1)}\n")

# Min/Max
print(f"Min: {X.min():.3f}, Max: {X.max():.3f}")
print(f"Argmin: {X.argmin()}, Argmax: {X.argmax()}")  # Flattened index

# Cumulative operations
a = np.array([1, 2, 3, 4, 5])
print(f"\nArray: {a}")
print(f"Cumsum: {np.cumsum(a)}")  # [1, 3, 6, 10, 15]
print(f"Cumprod: {np.cumprod(a)}")  # [1, 2, 6, 24, 120]

# Keepdims (important for broadcasting)
print(f"\nX.mean(axis=0) shape: {X.mean(axis=0).shape}")
print(f"X.mean(axis=0, keepdims=True) shape: {X.mean(axis=0, keepdims=True).shape}")

## 7. Memory Layout and Performance

In [ ]:
# C-order (row-major) vs F-order (column-major)
n = 5000

# C-order (default)
A_c = np.random.randn(n, n)
print(f"C-order flags:\n{A_c.flags}\n")

# F-order
A_f = np.asfortranarray(A_c)
print(f"F-order flags:\n{A_f.flags}\n")

# Performance difference: row vs column access
# C-order: fast row access, slow column access
# F-order: fast column access, slow row access

start = time.time()
for i in range(n):
    _ = A_c[i, :].sum()  # Row access
time_c_row = time.time() - start

start = time.time()
for i in range(n):
    _ = A_c[:, i].sum()  # Column access
time_c_col = time.time() - start

start = time.time()
for i in range(n):
    _ = A_f[:, i].sum()  # Column access
time_f_col = time.time() - start

print(f"C-order row access: {time_c_row:.4f}s")
print(f"C-order column access: {time_c_col:.4f}s")
print(f"F-order column access: {time_f_col:.4f}s")
print(f"\nC-order column access is {time_c_col/time_c_row:.2f}x slower than row access")

# Views vs Copies
a = np.arange(10)
b = a[2:5]  # View (slice)
c = a[[2, 3, 4]]  # Copy (fancy indexing)

print(f"\nOriginal: {a}")
b[0] = 999
print(f"After modifying view: {a}")  # a changed!
c[0] = 777
print(f"After modifying copy: {a}")  # a unchanged

# Force copy
d = a[2:5].copy()
d[0] = 111
print(f"After modifying explicit copy: {a}")

## 8. Practical ML Examples

In [ ]:
# Example 1: Softmax function
def softmax(x: np.ndarray) -> np.ndarray:
    """
    Numerically stable softmax.
    x: (batch_size, num_classes)
    """
    # Subtract max for numerical stability
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

logits = np.random.randn(3, 5)  # 3 samples, 5 classes
probs = softmax(logits)
print(f"Logits:\n{logits}\n")
print(f"Softmax probabilities:\n{probs}\n")
print(f"Row sums (should be 1): {probs.sum(axis=1)}\n")

# Example 2: One-hot encoding
def one_hot(labels: np.ndarray, num_classes: int) -> np.ndarray:
    """Convert labels to one-hot encoding."""
    n = labels.shape[0]
    one_hot = np.zeros((n, num_classes))
    one_hot[np.arange(n), labels] = 1
    return one_hot

labels = np.array([0, 2, 1, 0, 3])
one_hot_labels = one_hot(labels, num_classes=4)
print(f"Labels: {labels}")
print(f"One-hot:\n{one_hot_labels}\n")

# Example 3: Mini-batch generation
def create_mini_batches(X: np.ndarray, y: np.ndarray, batch_size: int):
    """Generate mini-batches for training."""
    n = X.shape[0]
    indices = np.random.permutation(n)
    X_shuffled = X[indices]
    y_shuffled = y[indices]
    
    for i in range(0, n, batch_size):
        yield X_shuffled[i:i+batch_size], y_shuffled[i:i+batch_size]

# Test
X_train = np.random.randn(100, 20)
y_train = np.random.randint(0, 10, size=100)

print("Mini-batches:")
for i, (X_batch, y_batch) in enumerate(create_mini_batches(X_train, y_train, batch_size=32)):
    print(f"Batch {i}: X.shape={X_batch.shape}, y.shape={y_batch.shape}")

## 9. Advanced Techniques

In [ ]:
# Einstein summation (einsum) - powerful notation
# Matrix multiplication: C_ij = sum_k A_ik * B_kj
A = np.random.randn(3, 4)
B = np.random.randn(4, 5)

C1 = A @ B
C2 = np.einsum('ik,kj->ij', A, B)
print(f"Matrix multiply via einsum: {np.allclose(C1, C2)}\n")

# Batch matrix multiplication
batch_A = np.random.randn(10, 3, 4)  # 10 matrices of 3x4
batch_B = np.random.randn(10, 4, 5)  # 10 matrices of 4x5
batch_C = np.einsum('bij,bjk->bik', batch_A, batch_B)  # (10, 3, 5)
print(f"Batch matmul shape: {batch_C.shape}\n")

# Trace (sum of diagonal)
A = np.random.randn(5, 5)
trace1 = np.trace(A)
trace2 = np.einsum('ii->', A)
print(f"Trace via einsum: {np.allclose(trace1, trace2)}\n")

# Outer product
a = np.array([1, 2, 3])
b = np.array([4, 5])
outer1 = np.outer(a, b)
outer2 = np.einsum('i,j->ij', a, b)
print(f"Outer product:\n{outer1}")
print(f"Einsum outer product:\n{outer2}\n")

# Diagonal extraction
A = np.arange(16).reshape(4, 4)
diag = np.einsum('ii->i', A)
print(f"Matrix:\n{A}")
print(f"Diagonal: {diag}")

## 10. Performance Tips Summary

1. **Vectorize**: Avoid explicit Python loops
2. **Broadcasting**: Use instead of tiling/repeating
3. **Views vs Copies**: Slicing creates views, fancy indexing creates copies
4. **Memory layout**: C-order for row operations, F-order for column operations
5. **In-place operations**: Use `+=`, `*=`, etc. to save memory
6. **Preallocate**: Create arrays once, fill in-place
7. **Use einsum**: For complex tensor operations
8. **Avoid repeated shape changes**: reshape/transpose are relatively cheap but still have overhead
9. **Profile**: Use `%timeit` in Jupyter or `time.time()` to measure
10. **Numba/Cython**: For truly performance-critical loops that can't be vectorized

## Exercises

### Exercise 1: Implement Batch Normalization
Implement batch normalization with learnable parameters γ (scale) and β (shift).

$$\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}$$
$$y = \gamma \hat{x} + \beta$$

In [ ]:
def batch_norm(X: np.ndarray, gamma: np.ndarray, beta: np.ndarray, 
               eps: float = 1e-8) -> np.ndarray:
    """
    Implement batch normalization.
    
    Args:
        X: (batch_size, features)
        gamma: (features,) - scale parameter
        beta: (features,) - shift parameter
        eps: numerical stability
    
    Returns:
        X_norm: (batch_size, features)
    """
    # TODO: Implement
    pass

# Test
X = np.random.randn(128, 64)
gamma = np.ones(64)
beta = np.zeros(64)
# X_norm = batch_norm(X, gamma, beta)
# print(f"Normalized mean: {X_norm.mean(axis=0)[:5]}")  # Should be ~0
# print(f"Normalized std: {X_norm.std(axis=0)[:5]}")    # Should be ~1

### Exercise 2: Implement K-Nearest Neighbors
Implement KNN classification without loops using vectorized distance computation.

In [ ]:
def knn_predict(X_train: np.ndarray, y_train: np.ndarray, 
                X_test: np.ndarray, k: int = 5) -> np.ndarray:
    """
    KNN classifier.
    
    Args:
        X_train: (n_train, d)
        y_train: (n_train,)
        X_test: (n_test, d)
        k: number of neighbors
    
    Returns:
        y_pred: (n_test,)
    """
    # TODO: Implement using vectorized distance computation
    pass

# Test
# X_train = np.random.randn(100, 10)
# y_train = np.random.randint(0, 3, size=100)
# X_test = np.random.randn(20, 10)
# y_pred = knn_predict(X_train, y_train, X_test, k=5)
# print(f"Predictions: {y_pred}")

### Exercise 3: Implement Efficient Attention Mechanism
Implement scaled dot-product attention without explicit loops.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
def scaled_dot_product_attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray) -> np.ndarray:
    """
    Scaled dot-product attention.
    
    Args:
        Q: (seq_len_q, d_k)
        K: (seq_len_k, d_k)
        V: (seq_len_v, d_v) where seq_len_v == seq_len_k
    
    Returns:
        output: (seq_len_q, d_v)
    """
    # TODO: Implement
    pass

# Test
# Q = np.random.randn(10, 64)  # 10 queries, 64 dims
# K = np.random.randn(20, 64)  # 20 keys, 64 dims
# V = np.random.randn(20, 64)  # 20 values, 64 dims
# output = scaled_dot_product_attention(Q, K, V)
# print(f"Attention output shape: {output.shape}")  # Should be (10, 64)

## Summary

You've learned:
- ✅ Array creation and manipulation
- ✅ Broadcasting mechanics
- ✅ Vectorization (10-100x speedups)
- ✅ Advanced indexing
- ✅ Linear algebra operations
- ✅ Memory layout optimization
- ✅ Practical ML implementations

**Next**: PyTorch fundamentals (tensors + autograd)